# ZigNet Zig 0.16.0 — QLoRA fine-tuning

This notebook fine-tunes a PEFT adapter for `fulgidus/zignet-qwen2.5-coder-7b`. It intentionally refuses to train on seed data or examples without compiler-validation evidence. Run cells from top to bottom after any runtime restart.

In [ ]:
# 1. Runtime, paths and environment
import os, json, hashlib, subprocess, sys, inspect
from pathlib import Path

ROOT = Path('/content/zignet-latest')
DATA_DIR = ROOT / 'data' / 'training' / 'v0.16.0'
OUTPUT_DIR = Path('/content/zignet-016-qlora-output')
MODEL_ID = 'fulgidus/zignet-qwen2.5-coder-7b'
MAX_SEQ_LENGTH = 1024
SEED = 42

if not Path('/content').exists():
    raise RuntimeError('This notebook is configured for Google Colab. Adapt ROOT and OUTPUT_DIR for another environment.')
print('Python:', sys.version)
try:
    print(subprocess.check_output(['nvidia-smi'], text=True))
except Exception as exc:
    print('GPU diagnostic:', exc)

In [ ]:
# 2. Install the training stack before creating any notebook state
# If pip reports that the runtime must restart, restart it and run all cells from the top.
%pip -q install -U "pyarrow>=19,<24" transformers datasets accelerate peft trl bitsandbytes safetensors huggingface_hub

In [ ]:
# 3. Fetch source data and create deterministic, group-isolated splits
if not ROOT.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/suissa/zignet-latest.git', str(ROOT)], check=True)
else:
    subprocess.run(['git', '-C', str(ROOT), 'pull', '--ff-only'], check=True)

DATA_DIR.mkdir(parents=True, exist_ok=True)
split_names = {'dataset-train.jsonl', 'dataset-validation.jsonl', 'dataset-test.jsonl'}
source_files = sorted(p for p in DATA_DIR.glob('*.jsonl') if p.name not in split_names and not p.name.startswith('.'))
if not source_files:
    raise FileNotFoundError(f'No category JSONL files found in {DATA_DIR}; refusing to use legacy data.')

records = []
for path in source_files:
    with path.open(encoding='utf-8') as handle:
        for line_no, line in enumerate(handle, 1):
            if not line.strip():
                continue
            try:
                row = json.loads(line)
            except json.JSONDecodeError as exc:
                raise ValueError(f'Invalid JSON at {path.name}:{line_no}: {exc}') from exc
            missing = {'instruction', 'input', 'output', 'metadata'} - set(row)
            if missing:
                raise ValueError(f'{path.name}:{line_no} missing fields: {sorted(missing)}')
            if not all(isinstance(row.get(k), str) for k in ('instruction', 'input', 'output')):
                raise ValueError(f'{path.name}:{line_no} instruction/input/output must be strings')
            if not isinstance(row.get('metadata'), dict):
                raise ValueError(f'{path.name}:{line_no} metadata must be an object')
            row['_source_file'] = path.name
            records.append(row)

if len(records) < 30:
    raise ValueError(f'Only {len(records)} examples found; refusing to create tiny splits.')

# Prefer explicit family IDs; otherwise group by source category. A family is never split across sets.
families = {}
for row in records:
    meta = row['metadata']
    family = meta.get('split_group') or meta.get('scenario_family') or meta.get('change_id') or row['_source_file']
    families.setdefault(str(family), []).append(row)
family_names = sorted(families)
if len(family_names) < 3:
    raise ValueError(f'Only {len(family_names)} independent groups; at least 3 are required for train/validation/test.')

# Stable hash assignment avoids depending on file enumeration or Python's randomized hash().
def bucket_for(name):
    value = int(hashlib.sha256(name.encode('utf-8')).hexdigest()[:8], 16) % 10
    return 'test' if value == 0 else 'validation' if value == 1 else 'train'

split_rows = {'train': [], 'validation': [], 'test': []}
for family in family_names:
    split_rows[bucket_for(family)].extend(families[family])

# Avoid empty sets when group counts are small; fail rather than silently leaking groups.
for name, rows in split_rows.items():
    if not rows:
        raise ValueError(f'The {name} split is empty. Add more independent scenario groups before training.')

for rows in split_rows.values():
    for row in rows:
        row.pop('_source_file', None)

split_paths = {
    'train': DATA_DIR / 'dataset-train.jsonl',
    'validation': DATA_DIR / 'dataset-validation.jsonl',
    'test': DATA_DIR / 'dataset-test.jsonl',
}
for name, path in split_paths.items():
    with path.open('w', encoding='utf-8') as handle:
        for row in split_rows[name]:
            handle.write(json.dumps(row, ensure_ascii=False) + '\n')

train_path = split_paths['train']
val_path = split_paths['validation']
test_path = split_paths['test']
print('Source categories:', len(source_files), '| total records:', len(records))
print('Split counts:', {name: len(rows) for name, rows in split_rows.items()})
print('Split groups:', {name: len({str((r.get('metadata') or {}).get('split_group') or (r.get('metadata') or {}).get('scenario_family') or (r.get('metadata') or {}).get('change_id') or 'unknown') for r in rows}) for name, rows in split_rows.items()})
compiler_validated = sum((r.get('metadata') or {}).get('compiler_validated') is True for r in records)
print(f'Compiler-validated records: {compiler_validated}/{len(records)}')
print('Note: creating splits does not validate code or make the dataset eligible for training.')

In [ ]:
# 4. Validate dataset schema, provenance and compiler evidence before loading a model
from datasets import load_dataset

data_files = {'train': str(train_path), 'validation': str(val_path), 'test': str(test_path)}
raw = load_dataset('json', data_files=data_files, cache_dir='/content/hf-datasets-cache')
required_columns = {'instruction', 'input', 'output', 'metadata'}
for split_name in ('train', 'validation', 'test'):
    dataset = raw[split_name]
    missing = required_columns - set(dataset.column_names)
    if missing:
        raise ValueError(f'{split_name}: missing columns {sorted(missing)}; found {dataset.column_names}')
    if len(dataset) == 0:
        raise ValueError(f'{split_name} split is empty')
    print(split_name, len(dataset), dataset.column_names)

def validation_errors(example):
    meta = example.get('metadata') or {}
    errors = []
    if meta.get('compiler_validated') is not True:
        errors.append('compiler_validated is not true')
    if meta.get('validation_status') not in {'compiler_validated', 'passed'}:
        errors.append('validation_status is not passed')
    if meta.get('compiler_version') != '0.16.0':
        errors.append('compiler_version is not pinned to 0.16.0')
    if not meta.get('validation_command'):
        errors.append('validation_command missing')
    if meta.get('validation_exit_code') != 0 and meta.get('validation_result') != 'passed':
        errors.append('successful compiler result evidence missing')
    if not meta.get('source') and not meta.get('source_url'):
        errors.append('source URL missing')
    if not (meta.get('change_id') or meta.get('scenario_family') or meta.get('scenario')):
        errors.append('scenario/change identifier missing')
    return errors

invalid = []
for split_name in ('train', 'validation', 'test'):
    for index, example in enumerate(raw[split_name]):
        errors = validation_errors(example)
        if errors:
            invalid.append((split_name, index, errors))
print(f'Examples missing required validation evidence: {len(invalid)}')
if invalid:
    for split_name, index, errors in invalid[:10]:
        print(f'  {split_name}[{index}]: ' + '; '.join(errors))
    raise RuntimeError(
        'TRAINING BLOCKED: the dataset is not compiler-validated with traceable evidence. '
        'Run scripts/audit_zig_training_dataset.py and fix the source records; do not bypass this gate.'
    )
print('Dataset quality gate passed. Model loading is now permitted.')

In [ ]:
# 5. Load the model in 4-bit and configure LoRA prerequisites
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training

if not torch.cuda.is_available():
    raise RuntimeError('CUDA GPU is unavailable. Select Runtime > Change runtime type > GPU.')
print('GPU:', torch.cuda.get_device_name(0))
compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=False)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=False,
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
print('Loaded:', MODEL_ID, '| dtype:', compute_dtype)

In [ ]:
# 6. Configure LoRA and format records for supervised fine-tuning
candidate_suffixes = {'q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'}
available = {
    name.split('.')[-1]
    for name, module in model.named_modules()
    if name.split('.')[-1] in candidate_suffixes and hasattr(module, 'weight')
}
target_modules = sorted(available)
if not target_modules:
    raise RuntimeError('No supported Qwen projection modules found. Inspect the checkpoint architecture before continuing.')
print('LoRA target modules:', target_modules)
lora_config = LoraConfig(
    r=8, lora_alpha=16, lora_dropout=0.05, bias='none',
    task_type='CAUSAL_LM', target_modules=target_modules,
)

def format_example(example):
    instruction = str(example['instruction']).strip()
    input_text = str(example['input']).strip()
    output_text = str(example['output']).strip()
    if not instruction or not output_text:
        raise ValueError('Every example must have a non-empty instruction and output.')
    user_text = instruction + ('\n\nInput:\n' + input_text if input_text else '')
    messages = [{'role': 'user', 'content': user_text}, {'role': 'assistant', 'content': output_text}]
    if getattr(tokenizer, 'chat_template', None):
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    else:
        text = f'### Instruction\n{instruction}\n\n### Input\n{input_text}\n\n### Response\n{output_text}{tokenizer.eos_token or ""}'
    return {'text': text}

formatted = raw.map(
    format_example,
    remove_columns=raw['train'].column_names,
    desc='Formatting examples',
)
for split_name in ('train', 'validation', 'test'):
    if 'text' not in formatted[split_name].column_names:
        raise RuntimeError(f'{split_name}: formatting did not produce the text column')
    if any(not text.strip() for text in formatted[split_name]['text']):
        raise RuntimeError(f'{split_name}: empty formatted text detected')
print('Formatted sample:\n', formatted['train'][0]['text'][:800])

In [ ]:
# 7. Build an SFTTrainer using the installed TRL/Transformers API
import transformers, trl
from transformers import TrainingArguments
from trl import SFTTrainer

print('transformers:', transformers.__version__, '| trl:', trl.__version__)
trainer_params = inspect.signature(SFTTrainer.__init__).parameters
base_training_kwargs = {
    'output_dir': str(OUTPUT_DIR),
    'num_train_epochs': 2,
    'per_device_train_batch_size': 1,
    'per_device_eval_batch_size': 1,
    'gradient_accumulation_steps': 16,
    'learning_rate': 1e-4,
    'lr_scheduler_type': 'cosine',
    'logging_steps': 5,
    'save_strategy': 'steps',
    'save_steps': 50,
    'save_total_limit': 2,
    'gradient_checkpointing': True,
    'fp16': compute_dtype == torch.float16,
    'bf16': compute_dtype == torch.bfloat16,
    'optim': 'paged_adamw_8bit',
    'report_to': 'none',
    'remove_unused_columns': False,
    'dataloader_num_workers': 0,
    'max_grad_norm': 0.3,
    'seed': SEED,
}
training_args_cls = None
try:
    from trl import SFTConfig
    training_args_cls = SFTConfig
except ImportError:
    training_args_cls = TrainingArguments

args_params = inspect.signature(training_args_cls.__init__).parameters
training_kwargs = {key: value for key, value in base_training_kwargs.items() if key in args_params}
if 'eval_strategy' in args_params:
    training_kwargs['eval_strategy'] = 'steps'
    training_kwargs['eval_steps'] = 50
elif 'evaluation_strategy' in args_params:
    training_kwargs['evaluation_strategy'] = 'steps'
    if 'eval_steps' in args_params:
        training_kwargs['eval_steps'] = 50
else:
    raise RuntimeError(f'{training_args_cls.__name__} exposes no evaluation strategy; installed package APIs are incompatible.')

if 'warmup_ratio' in args_params:
    training_kwargs['warmup_ratio'] = 0.03
elif 'warmup_steps' in args_params:
    training_kwargs['warmup_steps'] = 10

if training_args_cls.__name__ == 'SFTConfig':
    if 'dataset_text_field' in args_params:
        training_kwargs['dataset_text_field'] = 'text'
    if 'max_length' in args_params:
        training_kwargs['max_length'] = MAX_SEQ_LENGTH
    elif 'max_seq_length' in args_params:
        training_kwargs['max_seq_length'] = MAX_SEQ_LENGTH
    if 'packing' in args_params:
        training_kwargs['packing'] = False
args = training_args_cls(**training_kwargs)

trainer_kwargs = {
    'model': model,
    'args': args,
    'train_dataset': formatted['train'],
    'eval_dataset': formatted['validation'],
    'peft_config': lora_config,
}
if 'processing_class' in trainer_params:
    trainer_kwargs['processing_class'] = tokenizer
elif 'tokenizer' in trainer_params:
    trainer_kwargs['tokenizer'] = tokenizer
else:
    raise RuntimeError('SFTTrainer accepts neither processing_class nor tokenizer; inspect the installed TRL API.')

# Older TRL releases configure text fields on SFTTrainer rather than SFTConfig.
if training_args_cls.__name__ != 'SFTConfig':
    if 'dataset_text_field' in trainer_params:
        trainer_kwargs['dataset_text_field'] = 'text'
    if 'max_seq_length' in trainer_params:
        trainer_kwargs['max_seq_length'] = MAX_SEQ_LENGTH
    elif 'max_length' in trainer_params:
        trainer_kwargs['max_length'] = MAX_SEQ_LENGTH
    else:
        raise RuntimeError('Installed SFTTrainer has no supported maximum sequence-length parameter.')

unsupported_trainer = sorted(set(trainer_kwargs) - set(trainer_params))
if unsupported_trainer:
    raise RuntimeError(f'Unsupported SFTTrainer arguments: {unsupported_trainer}')
trainer = SFTTrainer(**trainer_kwargs)
if hasattr(trainer.model, 'print_trainable_parameters'):
    trainer.model.print_trainable_parameters()
print('Trainer initialized. Run the training cell only after the dataset gate passes.')

In [ ]:
# 8. Train the adapter
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
torch.cuda.reset_peak_memory_stats()
train_result = trainer.train()
print('Train metrics:', train_result.metrics)
print('Peak allocated VRAM GiB:', round(torch.cuda.max_memory_allocated() / 1024**3, 2))
print('Peak reserved VRAM GiB:', round(torch.cuda.max_memory_reserved() / 1024**3, 2))

In [ ]:
# 9. Evaluate validation and held-out test sets
validation_metrics = trainer.evaluate(eval_dataset=formatted['validation'], metric_key_prefix='validation')
test_metrics = trainer.evaluate(eval_dataset=formatted['test'], metric_key_prefix='test')
print('Validation metrics:', validation_metrics)
print('Test metrics:', test_metrics)

In [ ]:
# 10. Save adapter and reproducibility metadata
adapter_dir = OUTPUT_DIR / 'adapter'
trainer.model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
metadata = {
    'base_model': MODEL_ID,
    'dataset_dir': str(DATA_DIR),
    'dataset_counts': {name: len(dataset) for name, dataset in raw.items()},
    'max_seq_length': MAX_SEQ_LENGTH,
    'seed': SEED,
    'lora': {'r': 8, 'alpha': 16, 'dropout': 0.05, 'target_modules': target_modules},
    'compute_dtype': str(compute_dtype),
    'train_metrics': train_result.metrics,
    'validation_metrics': validation_metrics,
    'test_metrics': test_metrics,
    'transformers_version': transformers.__version__,
    'trl_version': trl.__version__,
    'torch_version': torch.__version__,
    'cuda_version': torch.version.cuda,
    'gpu': torch.cuda.get_device_name(0),
}
(OUTPUT_DIR / 'run-metadata.json').write_text(json.dumps(metadata, indent=2, default=str), encoding='utf-8')
print('Adapter and metadata saved under:', OUTPUT_DIR)

## Notes and safety gates

- The notebook does not mark data as validated. Every training example must have compiler-validation evidence produced by the repository's validation workflow.
- This creates a PEFT adapter, not merged weights or GGUF.
- On an RTX 4070 12 GB, start with `MAX_SEQ_LENGTH=512`, batch size 1 and a short pilot.
- The 1 vCPU / 1 GB RAM VPS is suitable for dataset auditing, not 7B fine-tuning.
- If package installation restarts the runtime, rerun every cell from the top.
- Do not remove the validation gate just to make training start; repair the data first.